# 03 — Xây dựng đặc trưng cơ sở và target ở cấp người chơi-trận

**Mục tiêu:** Tạo player_match_base có grain, lineage, công thức, target validity, dictionary, schema và partition manifest được kiểm chứng.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


Chọn `runtime` để chạy không cần Drive, hoặc `drive` để 13 notebook dùng chung dữ liệu bền vững. Với `drive`, mọi notebook phải dùng cùng `PUBG_DRIVE_PROJECT_ROOT` và chạy theo thứ tự.

**Chạy nhóm:** chủ thư mục chia sẻ `PUBG_Project` với quyền Editor. Mỗi thành viên thêm shortcut của chính thư mục đó vào My Drive, chọn `drive` và bật `PUBG_REQUIRE_EXISTING_PROJECT = True`. Mỗi người mount Drive của mình; kết quả phải nằm trong cùng thư mục gốc được chia sẻ. Chạy xong notebook, chờ file hiện trên Drive rồi bàn giao cho người tiếp theo; mỗi lần chỉ một người ghi. Người nhận chạy cell cấu hình, Bootstrap và khởi tạo của notebook tiếp theo. Biến trong RAM không được chuyển sang phiên mới; cell đang chạy dở có thể phải chạy lại. Xem `TEAM_DRIVE.md` để thiết lập và xác nhận đường dẫn.


In [ ]:
# @title Chọn nơi lưu dữ liệu { display-mode: "form" }
# @markdown `runtime`: không cần Drive, phù hợp notebook All-in-One.
# @markdown `drive`: lưu nối tiếp 13 notebook trong cùng thư mục Google Drive.
PUBG_STORAGE_MODE = "drive"  # @param ["runtime", "drive"]
PUBG_DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/PUBG_Project/Project_PUBG"  # @param {type:"string"}
# @markdown Nhóm dùng cùng thư mục đã chia sẻ: bật True để tránh tạo nhầm project riêng khi thiếu shortcut.
PUBG_REQUIRE_EXISTING_PROJECT = True  # @param {type:"boolean"}
# @markdown Số dòng mỗi batch khi đọc CSV trong ZIP; giảm nếu RAM ít. Không lấy mẫu dữ liệu.
PUBG_BATCH_ROWS = 50000  # @param {type:"integer"}
# Optional team label. Blank uses a unique runtime process ID.
PUBG_WRITER_ID = ""  # @param {type:"string"}
# Set True only after confirming the previous writer runtime has stopped.
PUBG_FORCE_STAGE_TAKEOVER = False  # @param {type:"boolean"}


In [ ]:
# Bootstrap: runtime mode needs no Drive; drive mode persists stage outputs.
import base64
import importlib.util
import io
import os
from pathlib import Path
import subprocess
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
PUBG_STORAGE_MODE = globals().get("PUBG_STORAGE_MODE", "runtime").strip().lower()
if PUBG_STORAGE_MODE not in {"runtime", "drive"}:
    raise ValueError("PUBG_STORAGE_MODE must be 'runtime' or 'drive'")

if PUBG_STORAGE_MODE == "drive":
    if not IN_COLAB:
        raise RuntimeError("Drive mode is available only on Google Colab")
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path(globals().get(
        "PUBG_DRIVE_PROJECT_ROOT", "/content/drive/MyDrive/PUBG_Project/Project_PUBG"
    )).expanduser().resolve()
    if globals().get("PUBG_REQUIRE_EXISTING_PROJECT", False) and not (
        (PROJECT_ROOT / "configs/data.yaml").is_file()
        and (PROJECT_ROOT / "src/utils/config.py").is_file()
    ):
        raise FileNotFoundError(
            "Shared project not found: " + str(PROJECT_ROOT)
            + ". Check Editor access and the PUBG_Project shortcut in My Drive. "
            "No private project was created."
        )
else:
    _candidates = ([Path("/content/Project_PUBG")] if IN_COLAB else
                   [Path.cwd(), *Path.cwd().parents])
    _candidates += [p / "Project_PUBG" for p in list(_candidates)]
    PROJECT_ROOT = next((p.resolve() for p in _candidates
                         if (p / "configs/data.yaml").is_file() and (p / "src/utils/config.py").is_file()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = (Path("/content") if IN_COLAB else Path.cwd()) / "Project_PUBG"

if not (PROJECT_ROOT / "configs/data.yaml").is_file() or not (PROJECT_ROOT / "src/utils/config.py").is_file():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    _bundle = zipfile.ZipFile(io.BytesIO(base64.b64decode('...')))
    for _entry in _bundle.infolist():
        _target = (PROJECT_ROOT / _entry.filename).resolve()
        if not _target.is_relative_to(PROJECT_ROOT.resolve()):
            raise ValueError("Invalid bundled path")
        if not _target.exists():
            _target.parent.mkdir(parents=True, exist_ok=True)
            _target.write_bytes(_bundle.read(_entry))
    _bundle.close()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if globals().get("PUBG_INSTALL_DEPENDENCIES", IN_COLAB) and not globals().get("_PUBG_PACKAGES_READY", False):
    _requirements = {
        "numpy": "numpy>=1.24.0", "pandas": "pandas>=2.0.0",
        "pyarrow": "pyarrow>=12.0.0", "duckdb": "duckdb>=0.9.0",
        "scipy": "scipy>=1.10.0", "sklearn": "scikit-learn>=1.3.0",
        "yaml": "pyyaml>=6.0",
    }
    _missing = [spec for module, spec in _requirements.items() if importlib.util.find_spec(module) is None]
    if _missing:
        print("Installing missing packages:", ", ".join(_missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--prefer-binary", *_missing])
    _PUBG_PACKAGES_READY = True

if PUBG_STORAGE_MODE == "drive":
    os.environ["PUBG_SESSION_DRIVE_ROOT"] = str(PROJECT_ROOT)
    os.environ["PUBG_SESSION_TEMP_DIR"] = str(globals().get("PUBG_RUNTIME_TEMP_DIR", "/content/temp"))
else:
    os.environ.pop("PUBG_SESSION_DRIVE_ROOT", None)
    os.environ.pop("PUBG_SESSION_TEMP_DIR", None)
from src.utils.config import load_config, resolve_paths
cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
for _path in paths.values():
    _path.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)
_PUBG_CELL_PROGRESS = {}  # Re-bootstrap invalidates prior cell state, even within one kernel.
print("Storage:", paths["data_root"], "| Results:", paths["reports_root"])
if PUBG_STORAGE_MODE == "drive":
    print("Storage mode: Google Drive. Stage outputs persist for the next notebook.")
else:
    print("Storage mode: runtime. No Drive authorization required; export before reset.")


### 0. Bối cảnh khoa học, phạm vi và tiêu chí thành công

Notebook 03 chuyển dữ liệu sạch ở cấp người chơi-trận thành tập đặc trưng cơ sở dùng chung cho RQ1, RQ2 và RQ3. Đây là bước xây dữ liệu, chưa phải bước chứng minh quan hệ, phân cụm hay huấn luyện mô hình.

Đơn vị quan sát là một người chơi trong một trận. row_id là khóa kỹ thuật chống va chạm; khóa nghiệp vụ là (match_id, player_name). Nếu tên người chơi không khả dụng, lineage (source_file, source_row) chỉ dùng để truy vết, không biến một danh tính chưa biết thành người chơi đã xác minh.

Các mục tiêu kiểm chứng:

1. Bảo toàn đúng số dòng từ cleaned_aggregate.parquet, không join explosion.
2. Tính một lần các đặc trưng combat, movement và support tại nguồn canonical.
3. Giữ raw outcome, target chuẩn hóa và cờ hợp lệ riêng biệt.
4. Không che placement lỗi bằng clipping; không biến structural missing thành 0.
5. Tách team_size_mode khỏi perspective_mode; mode chưa xác minh nhận unknown.
6. Xuất dictionary, validation summary, schema và numbered parts có đối soát.
7. Chứng minh allowlist chặn feature target-derived khỏi task không phù hợp.

Notebook không lấy mẫu để tạo số liệu chính thức. Lấy mẫu có seed chỉ dùng cho hình phân bố và luôn ghi rõ N. Kết quả mô tả là association, không phải quan hệ nhân quả.


### 1. Tiền điều kiện, provenance và phạm vi dữ liệu

Notebook 03 yêu cầu ba hợp đồng từ Notebook 02: dữ liệu sạch, metadata roster và split đã khóa. Split chỉ được kiểm tra coverage ở bước này; dữ liệu base chưa dùng test để lựa chọn feature hay mô hình.

Cách đọc Bảng 03-A: mỗi artifact phải tồn tại, có kích thước lớn hơn 0 và checksum. Nếu thiếu, chạy lại Notebook 02 thay vì tự tạo dữ liệu thay thế.


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['03_build_player_match.ipynb'] = -1
from IPython.display import display, Image
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
from src.utils.config import load_config, resolve_paths
from src.data.io import get_duckdb_connection, read_json
from src.data.checkpoints import CheckpointManager
from src.utils.hashing import hash_file

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
if "con" in globals():
    con.close()
con = get_duckdb_connection(
    temp_dir=paths["temp_dir"],
    **{key: cfg["runtime"]["duckdb"][key] for key in ("memory_limit", "threads")},
)
ckpt_mgr = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")

cleaned_agg_pq = paths["interim"] / "cleaned_aggregate.parquet"
meta_pq = paths["interim"] / "match_metadata.parquet"
split_pq = paths["interim"] / "split_assignments.parquet"
_required_inputs = {
    "cleaned_aggregate": cleaned_agg_pq,
    "match_metadata": meta_pq,
    "split_assignments": split_pq,
}
_missing_inputs = [name for name, path in _required_inputs.items() if not path.is_file()]
if _missing_inputs:
    raise FileNotFoundError(
        f"Thiếu đầu vào Gate G2: {_missing_inputs}. Chạy hoàn tất Notebook 02 trước."
    )

_input_rows = []
for _name, _path in _required_inputs.items():
    _input_rows.append({
        "Artifact": _name,
        "Đường dẫn": str(_path.relative_to(PROJECT_ROOT)),
        "Bytes": _path.stat().st_size,
        "SHA256": hash_file(_path)[:16] + "...",
        "Phạm vi": "full cleaned data" if _name == "cleaned_aggregate" else "match-level contract",
        "Trạng thái": "Đạt",
    })
_df_inputs = pd.DataFrame(_input_rows)
print("--- BẢNG 03-A: TIỀN ĐIỀU KIỆN VÀ PROVENANCE ---")
print(_df_inputs.to_string(index=False))
display(_df_inputs)

_pubg_figure_details = {}
from src.data.checkpoints import CheckpointManager
from src.utils.hashing import compute_stage_signature, hash_file
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_manifest = _pubg_checkpoint.load_manifest()
_pubg_dependencies = ['02_data_quality_and_structure.ipynb']
_pubg_dependency_records = {name: {"signature": _pubg_manifest.get("stages", {}).get("notebook/" + name, {}).get("signature"), "checksums": _pubg_manifest.get("stages", {}).get("notebook/" + name, {}).get("checksums", {})} for name in _pubg_dependencies}
_pubg_code_files = sorted((PROJECT_ROOT / "src").rglob("*.py"))
_pubg_config_files = sorted((PROJECT_ROOT / "configs").glob("*.yaml"))
_pubg_signature = compute_stage_signature('03_build_player_match.ipynb', data_files=_pubg_config_files, config={"dependencies": _pubg_dependency_records}, code_files=_pubg_code_files, cohort=_pubg_dependency_records, split=cfg.get("rq3", {}).get("split"), registry=hash_file(PROJECT_ROOT / "src/models/registry.py"), feature_set=hash_file(PROJECT_ROOT / "configs/features.yaml"), backend={"rq2": cfg.get("rq2", {}).get("device"), "rq3": cfg.get("rq3", {}).get("device")})
_pubg_checkpoint.begin_notebook('03_build_player_match.ipynb', _pubg_dependencies, signature=_pubg_signature, writer_id=globals().get("PUBG_WRITER_ID") or None, force_takeover=bool(globals().get("PUBG_FORCE_STAGE_TAKEOVER", False)))
_pubg_writer = _pubg_checkpoint.load_manifest()["stages"]['notebook/03_build_player_match.ipynb']["writer_id"]
PUBG_FORCE_STAGE_TAKEOVER = False  # Tiếp quản một lần; không tự ép lần sau.

_pubg_progress['03_build_player_match.ipynb'] = 6


### 2. Feature Registry và hợp đồng chống leakage

Feature Registry là từ điển có thẩm quyền. Mỗi feature ghi tên, nhóm, nguồn, cấp dữ liệu, công thức, đơn vị, kiểu dữ liệu, mẫu số, missing semantics, allowlist và cấm theo target.

Ba quy tắc quan trọng:

- ride_ratio không tồn tại như main feature vì gần bằng 1 - walk_ratio.
- dbno_ratio và feature tổng hợp cấp đội không tự động vào core.
- Rate chứa player_survive_time chỉ dùng diagnostic; không dùng cho chính survival target.

Bảng 03-B kiểm kê 26 feature không historical cùng các khai báo lịch sử dành cho NB08;
khai báo trong dictionary không có nghĩa feature lịch sử đã được tính ở NB03.
Bảng 03-C tập trung vào công thức cơ sở; Bảng 03-D cho biết task nào được phép dùng từng feature.


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('03_build_player_match.ipynb', -1) < 6:
    raise RuntimeError("03_build_player_match.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['03_build_player_match.ipynb'] = 6
try:
    from IPython.display import display, Image
    from src.features.registry import FeatureRegistry

    dict_csv = paths["tables"] / "feature_dictionary.csv"
    registry = FeatureRegistry()
    registry.export_dictionary_csv(dict_csv)
    df_dict = pd.read_csv(dict_csv)

    _df_registry_summary = (
        df_dict.groupby(["group", "status"], dropna=False)
        .size().reset_index(name="Số feature")
        .sort_values(["group", "status"])
    )
    print("--- BẢNG 03-B: KIỂM KÊ FEATURE REGISTRY ---")
    print(_df_registry_summary.to_string(index=False))
    display(_df_registry_summary)
    print(f"Tổng số feature đăng ký: {len(df_dict)}")

    _core_names = [
        "damage_per_kill", "total_distance", "walk_ratio", "assist_ratio",
        "player_survive_time", "normalized_placement",
    ]
    _df_core_dictionary = df_dict[df_dict["feature_name"].isin(_core_names)][[
        "feature_name", "source_columns", "formula", "unit", "denominator",
        "missing_semantics", "allowed_tasks", "forbidden_targets",
    ]]
    print("\n--- BẢNG 03-C: CÔNG THỨC, ĐƠN VỊ VÀ MISSING SEMANTICS ---")
    print(_df_core_dictionary.to_string(index=False))
    display(_df_core_dictionary)

    _tasks = ["rq1_survival", "rq1_placement", "rq2", "s1", "p1", "p2", "diagnostic"]
    _allowlist_rows = []
    for _task in _tasks:
        _allowed = registry.get_allowed_features(_task)
        _allowlist_rows.append({
            "Task": _task,
            "Số feature được phép": len(_allowed),
            "Danh sách": ", ".join(_allowed),
            "Có rate theo survival": any(name in _allowed for name in ("kills_per_minute", "damage_per_minute")),
        })
    _df_allowlists = pd.DataFrame(_allowlist_rows)
    print("\n--- BẢNG 03-D: TASK ALLOWLIST VÀ KIỂM SOÁT LEAKAGE ---")
    print(_df_allowlists.to_string(index=False))
    display(_df_allowlists)

except (Exception, KeyboardInterrupt) as _nb02_error:
    _pubg_progress['03_build_player_match.ipynb'] = -1
    try:
        import duckdb
        _pubg_checkpoint.record_failure('notebook/03_build_player_match.ipynb', _pubg_signature, str(_nb02_error), resource_limited=isinstance(_nb02_error, (MemoryError, duckdb.OutOfMemoryException)), writer_id=_pubg_writer)
    except Exception as _nb02_checkpoint_error:
        print("Không ghi được trạng thái lỗi; giữ lỗi gốc và kiểm tra manifest trước khi tiếp quản:", _nb02_checkpoint_error)
    raise


_pubg_progress['03_build_player_match.ipynb'] = 8


### 3. Tạo player_match_base và báo cáo validation

Các công thức:

[
damage_per_kill = rac{player_dmg}{player_kills},
qquad
walk_ratio = rac{player_dist_walk}{player_dist_walk+player_dist_ride},
]

[
assist_ratio = rac{player_assists}{player_assists+player_kills},
qquad
normalized_placement =
1-rac{team_placement-1}{N_{teams}-1}.
]

Mẫu số bằng 0 tạo structural missing (NULL), không dùng epsilon và không gán 0. Placement chỉ được tính khi roster hoàn chỉnh, N_teams > 1 và placement thuộc [1,N_teams].

Output canonical là player_match_base.parquet. Thư mục numbered parts hỗ trợ đọc theo phần nhưng không tạo thư mục riêng cho từng player hoặc match.


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('03_build_player_match.ipynb', -1) < 8:
    raise RuntimeError("03_build_player_match.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['03_build_player_match.ipynb'] = 8
try:
    from IPython.display import display, Image
    from src.features.base import build_player_match_base

    base_pq = paths["interim"] / "player_match_base.parquet"
    val_csv = paths["tables"] / "feature_validation_base.csv"
    schema_json = paths["manifests"] / "player_match_base_schema.json"
    parts_dir = paths["interim"] / "player_match_base_parts"
    parts_manifest_json = paths["manifests"] / "player_match_base_parts_manifest.json"

    base_rows = build_player_match_base(
        con=con,
        cleaned_aggregate_parquet=cleaned_agg_pq,
        match_metadata_parquet=meta_pq,
        output_base_parquet=base_pq,
        validation_csv_path=val_csv,
        dictionary_csv_path=dict_csv,
        split_assignments_parquet=split_pq,
        schema_json_path=schema_json,
        partition_dir=parts_dir,
        partition_manifest_path=parts_manifest_json,
    )

    df_val = pd.read_csv(val_csv)
    print("--- BẢNG 03-E: VALIDATION VÀ PHÂN LOẠI GIÁ TRỊ THIẾU ---")
    print(df_val[[
        "feature_name", "feature_group", "total_rows", "valid_count", "null_count",
        "structural_missing_count", "other_missing_count", "null_pct", "unit", "status",
    ]].to_string(index=False))
    display(df_val[[
        "feature_name", "feature_group", "total_rows", "valid_count", "null_count",
        "structural_missing_count", "other_missing_count", "null_pct", "unit", "status",
    ]])

    _n_clean = con.execute("SELECT count(*) FROM read_parquet(?)", [str(cleaned_agg_pq)]).fetchone()[0]
    _reconciliation = pd.DataFrame([
        {"Đại lượng": "Rows cleaned đầu vào", "Kỳ vọng": _n_clean, "Thực tế": _n_clean, "Kết luận": "Đạt"},
        {"Đại lượng": "Rows player_match_base", "Kỳ vọng": _n_clean, "Thực tế": base_rows, "Kết luận": "Đạt" if base_rows == _n_clean else "Không đạt"},
    ])
    print("\n--- BẢNG 03-F: ĐỐI SOÁT BẢO TOÀN DÒNG ---")
    print(_reconciliation.to_string(index=False))
    display(_reconciliation)
    if base_rows != _n_clean:
        raise RuntimeError(f"Join explosion hoặc mất dòng: base={base_rows}, clean={_n_clean}")

except (Exception, KeyboardInterrupt) as _nb02_error:
    _pubg_progress['03_build_player_match.ipynb'] = -1
    try:
        import duckdb
        _pubg_checkpoint.record_failure('notebook/03_build_player_match.ipynb', _pubg_signature, str(_nb02_error), resource_limited=isinstance(_nb02_error, (MemoryError, duckdb.OutOfMemoryException)), writer_id=_pubg_writer)
    except Exception as _nb02_checkpoint_error:
        print("Không ghi được trạng thái lỗi; giữ lỗi gốc và kiểm tra manifest trước khi tiếp quản:", _nb02_checkpoint_error)
    raise


_pubg_progress['03_build_player_match.ipynb'] = 10


### 4. Grain, schema và numbered parts

row_id phải không null và duy nhất. Một player name rỗng chỉ được giữ khi có lineage đầy đủ; điều này bảo toàn truy vết nhưng không xác nhận danh tính.

Schema manifest khóa tên và kiểu cột. Parts manifest ghi partition rule, số dòng, bytes và checksum từng part. Tổng số dòng của các part phải đúng bằng file canonical.

**Tính toàn vẹn Parquet:** footer và checksum đúng chưa chứng minh dữ liệu nén đọc được. Producer đọc giải nén toàn bộ cột theo batch bằng PyArrow và quét bằng DuckDB trước khi công bố, sau đó kiểm tra lại file đích. Kiểm tra này tăng I/O nhưng không lấy mẫu hay bỏ dòng. Nếu báo ZSTD Decompression failure, dừng, kiểm tra kết nối/storage và không commit NB03. Không đổi codec hoặc sửa manifest để che lỗi. Sau khi sửa storage, khởi tạo lại NB03 rồi chạy lại các cell từ registry tới tạo base và các cell kiểm toán; không cần chạy lại NB01/NB02 nếu upstream vẫn completed và dữ liệu hợp lệ. NB03 sẽ tạo lại base/parts đúng tên; không dùng bản lỗi làm đầu vào NB04.

Nếu checkpoint còn running từ runtime đã mất, chỉ bật PUBG_FORCE_STAGE_TAKEOVER khi đã xác nhận runtime cũ ngắt và không ai đang ghi. Các lỗi xử lý NB03 được ghi failed/resource_limited nếu Drive còn ghi được; cần khởi tạo lại trước retry.


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('03_build_player_match.ipynb', -1) < 10:
    raise RuntimeError("03_build_player_match.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['03_build_player_match.ipynb'] = 10
try:
    from IPython.display import display, Image
    _schema = read_json(schema_json)
    _parts = read_json(parts_manifest_json)
    _grain = con.execute('''
        SELECT
            count(*) AS rows,
            count(DISTINCT row_id) AS unique_row_ids,
            count(*) FILTER (WHERE row_id IS NULL) AS null_row_ids,
            count(DISTINCT match_id) AS matches,
            count(DISTINCT player_name) AS named_players,
            count(*) FILTER (WHERE player_name IS NULL OR length(trim(player_name)) = 0) AS lineage_fallback_rows
        FROM read_parquet(?)
    ''', [str(base_pq)]).df()
    print("--- BẢNG 03-G1: GRAIN VÀ KHÓA DÒNG ---")
    print(_grain.to_string(index=False))
    display(_grain)

    _df_schema = pd.DataFrame(_schema["columns"])
    print("\n--- BẢNG 03-G2: SCHEMA PLAYER_MATCH_BASE ---")
    print(_df_schema.to_string(index=False))
    display(_df_schema)

    _df_parts = pd.DataFrame(_parts["parts"])
    print("\n--- BẢNG 03-G3: NUMBERED PARTS VÀ CHECKSUM ---")
    print(_df_parts.to_string(index=False))
    display(_df_parts)
    print(
        f"Đối soát parts: canonical={_parts['row_count']:,}, "
        f"tổng parts={_parts['row_count_from_parts']:,}, số parts={_parts['part_count']}"
    )

except (Exception, KeyboardInterrupt) as _nb02_error:
    _pubg_progress['03_build_player_match.ipynb'] = -1
    try:
        import duckdb
        _pubg_checkpoint.record_failure('notebook/03_build_player_match.ipynb', _pubg_signature, str(_nb02_error), resource_limited=isinstance(_nb02_error, (MemoryError, duckdb.OutOfMemoryException)), writer_id=_pubg_writer)
    except Exception as _nb02_checkpoint_error:
        print("Không ghi được trạng thái lỗi; giữ lỗi gốc và kiểm tra manifest trước khi tiếp quản:", _nb02_checkpoint_error)
    raise


_pubg_progress['03_build_player_match.ipynb'] = 12


### 5. Ví dụ tính tay và kiểm tra target validity

Bảng 03-H đối chiếu công thức tính tay với output. Sai số phải gần 0. Các dòng có mẫu số 0 có kỳ vọng NULL.

Bảng 03-I không loại toàn bộ một dòng chỉ vì một target lỗi. valid_survival và valid_placement tạo hai cohort task-specific. Raw outcome vẫn được giữ để kiểm toán.


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('03_build_player_match.ipynb', -1) < 12:
    raise RuntimeError("03_build_player_match.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['03_build_player_match.ipynb'] = 12
try:
    from IPython.display import display, Image
    _examples = con.execute('''
        SELECT
            row_id, player_kills, player_dmg, damage_per_kill,
            CASE WHEN player_kills > 0 THEN player_dmg / player_kills ELSE NULL END AS damage_manual,
            total_distance, player_dist_walk, walk_ratio,
            CASE WHEN total_distance > 0 THEN player_dist_walk / total_distance ELSE NULL END AS walk_manual,
            player_assists, assist_ratio,
            CASE WHEN player_assists + player_kills > 0
                 THEN CAST(player_assists AS DOUBLE) / (player_assists + player_kills)
                 ELSE NULL END AS assist_manual,
            team_placement, observed_team_count, normalized_placement,
            CASE WHEN valid_placement
                 THEN 1.0 - CAST(team_placement - 1 AS DOUBLE) / (observed_team_count - 1)
                 ELSE NULL END AS placement_manual
        FROM read_parquet(?)
        ORDER BY row_id
        LIMIT 8
    ''', [str(base_pq)]).df()
    for _stored, _manual, _error in (
        ("damage_per_kill", "damage_manual", "damage_abs_error"),
        ("walk_ratio", "walk_manual", "walk_abs_error"),
        ("assist_ratio", "assist_manual", "assist_abs_error"),
        ("normalized_placement", "placement_manual", "placement_abs_error"),
    ):
        _examples[_error] = (_examples[_stored] - _examples[_manual]).abs()
    print("--- BẢNG 03-H: ĐỐI CHIẾU TÍNH TAY VỚI OUTPUT ---")
    print(_examples.to_string(index=False))
    display(_examples)

    _task_coverage = con.execute('''
        SELECT
            count(*) AS all_cleaned_rows,
            count(*) FILTER (WHERE valid_survival) AS survival_eligible_rows,
            count(*) FILTER (WHERE valid_placement) AS placement_eligible_rows,
            count(*) FILTER (WHERE valid_survival AND valid_placement) AS eligible_both,
            count(*) FILTER (WHERE valid_survival AND NOT valid_placement) AS survival_only,
            count(*) FILTER (WHERE valid_placement AND NOT valid_survival) AS placement_only,
            count(*) FILTER (WHERE NOT valid_survival AND NOT valid_placement) AS eligible_neither
        FROM read_parquet(?)
    ''', [str(base_pq)]).df()
    print("\n--- BẢNG 03-I: COVERAGE VÀ VALIDITY THEO TASK ---")
    print(_task_coverage.to_string(index=False))
    display(_task_coverage)

except (Exception, KeyboardInterrupt) as _nb02_error:
    _pubg_progress['03_build_player_match.ipynb'] = -1
    try:
        import duckdb
        _pubg_checkpoint.record_failure('notebook/03_build_player_match.ipynb', _pubg_signature, str(_nb02_error), resource_limited=isinstance(_nb02_error, (MemoryError, duckdb.OutOfMemoryException)), writer_id=_pubg_writer)
    except Exception as _nb02_checkpoint_error:
        print("Không ghi được trạng thái lỗi; giữ lỗi gốc và kiểm tra manifest trước khi tiếp quản:", _nb02_checkpoint_error)
    raise


_pubg_progress['03_build_player_match.ipynb'] = 14


### 6. Mode audit và giới hạn diễn giải

perspective_mode mô tả góc nhìn; team_size_mode mô tả quy mô đội. Hai khái niệm không được gộp thành game_mode mơ hồ.

Quy tắc hiện tại chỉ ánh xạ party_size đã xác minh: 1 là solo, 2 là duo, 4 là squad. Giá trị khác hoặc thiếu nhận unknown; Notebook 05 sẽ dùng EDA dữ liệu thật để quyết định chiến lược mode. Tỷ lệ mode chỉ mô tả cohort, không chứng minh mode gây ra khác biệt outcome.


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('03_build_player_match.ipynb', -1) < 14:
    raise RuntimeError("03_build_player_match.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['03_build_player_match.ipynb'] = 14
try:
    from IPython.display import display, Image
    _df_modes = con.execute('''
        SELECT
            party_size, team_size_mode, perspective_mode,
            count(*) AS rows,
            count(DISTINCT match_id) AS matches,
            round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS row_pct
        FROM read_parquet(?)
        GROUP BY party_size, team_size_mode, perspective_mode
        ORDER BY rows DESC, party_size
    ''', [str(base_pq)]).df()
    print("--- BẢNG 03-J: PARTY SIZE, TEAM SIZE MODE VÀ PERSPECTIVE ---")
    print(_df_modes.to_string(index=False))
    display(_df_modes)

except (Exception, KeyboardInterrupt) as _nb02_error:
    _pubg_progress['03_build_player_match.ipynb'] = -1
    try:
        import duckdb
        _pubg_checkpoint.record_failure('notebook/03_build_player_match.ipynb', _pubg_signature, str(_nb02_error), resource_limited=isinstance(_nb02_error, (MemoryError, duckdb.OutOfMemoryException)), writer_id=_pubg_writer)
    except Exception as _nb02_checkpoint_error:
        print("Không ghi được trạng thái lỗi; giữ lỗi gốc và kiểm tra manifest trước khi tiếp quản:", _nb02_checkpoint_error)
    raise


_pubg_progress['03_build_player_match.ipynb'] = 16


### 7. Trực quan kiểm toán

- V03-01: phân bố bốn feature dẫn xuất; chỉ dùng reservoir sample có seed 42, tối đa 100.000 dòng.
- V03-02: số dòng theo team_size_mode và perspective_mode trên toàn bộ dữ liệu.
- V03-03: coverage full-data của hai target; mẫu số là toàn bộ cleaned rows.

Histogram không chứng minh quan hệ nhân quả. Giá trị cực trị hợp lệ không bị xóa chỉ để hình đẹp; trục ghi đúng đơn vị và N.


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('03_build_player_match.ipynb', -1) < 16:
    raise RuntimeError("03_build_player_match.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['03_build_player_match.ipynb'] = 16
try:
    from IPython.display import display, Image
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig_dir = paths["figures"]
    fig_dir.mkdir(parents=True, exist_ok=True)
    _sample_n = min(base_rows, 100000)
    _df_sample = con.execute(f'''
        SELECT damage_per_kill, walk_ratio, assist_ratio, normalized_placement
        FROM read_parquet(?)
        USING SAMPLE {_sample_n} ROWS (reservoir, 42)
    ''', [str(base_pq)]).df()

    _fig, _axes = plt.subplots(2, 2, figsize=(12, 8))
    _feature_plot_meta = [
        ("damage_per_kill", "Damage trên mỗi kill", "đơn vị sát thương nguồn / kill; chưa xác minh"),
        ("walk_ratio", "Tỷ lệ quãng đường đi bộ", "tỷ lệ [0,1]"),
        ("assist_ratio", "Tỷ lệ hỗ trợ", "tỷ lệ [0,1]"),
        ("normalized_placement", "Thứ hạng chuẩn hóa", "điểm [0,1]"),
    ]
    for _ax, (_column, _title, _unit) in zip(_axes.flat, _feature_plot_meta):
        _values = _df_sample[_column].dropna()
        if len(_values):
            _ax.hist(_values, bins=30, color="#4C72B0", edgecolor="white")
        _ax.set_title(_title)
        _ax.set_xlabel(_unit)
        _ax.set_ylabel("Số dòng trong mẫu")
        _ax.grid(axis="y", alpha=0.25)
    _fig.suptitle(f"V03-01. Phân bố feature cơ sở; reservoir sample N={len(_df_sample):,}, seed=42")
    _fig.tight_layout()
    _fig_features = fig_dir / "nb03_feature_distributions.png"
    _pubg_figure_details = {_fig_features.name: {"source_table": str(base_pq), "scope": "base descriptive",
        "sample_n": len(_df_sample), "sample_seed": 42, "sampling_rule": "DuckDB reservoir; visualization only",
        "caption": f"V03-01: n={len(_df_sample)}, N base={base_rows}; missing bị loại từng panel, không loại khỏi dữ liệu."}}
    _fig.savefig(_fig_features, dpi=180, bbox_inches="tight")
    display(Image(filename=str(_fig_features)))
    plt.close(_fig)
    print(f"HÌNH V03-01: {_fig_features.name} | N mẫu={len(_df_sample):,} | chỉ dùng cho trực quan")

    _mode_plot = _df_modes.pivot_table(
        index="team_size_mode", columns="perspective_mode", values="rows",
        aggfunc="sum", fill_value=0,
    )
    _fig, _ax = plt.subplots(figsize=(9, 5))
    _mode_plot.plot(kind="bar", stacked=True, ax=_ax)
    _ax.set_title(f"V03-02. Phân bố mode trên toàn bộ base; N={base_rows:,}")
    _ax.set_xlabel("Team size mode")
    _ax.set_ylabel("Số dòng người chơi-trận")
    _ax.legend(title="Perspective mode")
    _ax.grid(axis="y", alpha=0.25)
    _fig.tight_layout()
    _fig_modes = fig_dir / "nb03_mode_distribution.png"
    _fig.savefig(_fig_modes, dpi=180, bbox_inches="tight")
    display(Image(filename=str(_fig_modes)))
    plt.close(_fig)
    print(f"HÌNH V03-02: {_fig_modes.name} | N base={base_rows:,} | runtime.mode={cfg['runtime']['mode']}; không tự chứng nhận full-data")

    _cov = _task_coverage.iloc[0]
    _cov_names = ["survival_eligible_rows", "placement_eligible_rows", "eligible_both"]
    _fig, _ax = plt.subplots(figsize=(9, 5))
    _bars = _ax.bar(
        ["Survival hợp lệ", "Placement hợp lệ", "Cả hai hợp lệ"],
        [_cov[name] for name in _cov_names],
        color=["#4C72B0", "#55A868", "#8172B2"],
    )
    _ax.set_title(f"V03-03. Coverage target theo cùng mẫu số N={base_rows:,}")
    _ax.set_ylabel("Số dòng người chơi-trận")
    _ax.set_ylim(0, max(base_rows, 1) * 1.12)
    for _bar in _bars:
        _ax.text(_bar.get_x() + _bar.get_width()/2, _bar.get_height(), f"{int(_bar.get_height()):,}", ha="center", va="bottom")
    _ax.grid(axis="y", alpha=0.25)
    _fig.tight_layout()
    _fig_coverage = fig_dir / "nb03_task_coverage.png"
    _fig.savefig(_fig_coverage, dpi=180, bbox_inches="tight")
    display(Image(filename=str(_fig_coverage)))
    plt.close(_fig)
    print(f"HÌNH V03-03: {_fig_coverage.name} | mẫu số={base_rows:,} cleaned rows")

except (Exception, KeyboardInterrupt) as _nb02_error:
    _pubg_progress['03_build_player_match.ipynb'] = -1
    try:
        import duckdb
        _pubg_checkpoint.record_failure('notebook/03_build_player_match.ipynb', _pubg_signature, str(_nb02_error), resource_limited=isinstance(_nb02_error, (MemoryError, duckdb.OutOfMemoryException)), writer_id=_pubg_writer)
    except Exception as _nb02_checkpoint_error:
        print("Không ghi được trạng thái lỗi; giữ lỗi gốc và kiểm tra manifest trước khi tiếp quản:", _nb02_checkpoint_error)
    raise


_pubg_progress['03_build_player_match.ipynb'] = 18


### 8. Kiểm tra hoàn tất NB03, giới hạn và bàn giao

NB03 hoàn tất khi row count được bảo toàn, row_id duy nhất, schema và parts đối soát, dictionary đủ metadata, công thức tính tay khớp và allowlist không chứa rate survival-derived trong S1. Đây không phải Gate G3: G3 dành cho quyết định nghiên cứu sau diagnostics/EDA, không được chứng nhận chỉ bằng kiểm tra base features.

Giới hạn:

- Đây là nghiệm thu code và fixture; bảng full-data chỉ hình thành khi chạy trên Drive.
- Mode unknown cần Notebook 05 mô tả trước khi chốt chiến lược phân tầng.
- Missing ngoài structural missing được giữ và báo cáo, không tự động điền 0.
- Notebook này dùng CPU và DuckDB; GPU không mặc định làm các phép SQL nhanh hơn.
- Notebook 04 chỉ nối timing vào base đã khóa, không tính lại feature cơ sở.


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('03_build_player_match.ipynb', -1) < 18:
    raise RuntimeError("03_build_player_match.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['03_build_player_match.ipynb'] = 18
try:
    from IPython.display import display, Image
    _error_columns = ["damage_abs_error", "walk_abs_error", "assist_abs_error", "placement_abs_error"]
    _finite_errors = []
    for _column in _error_columns:
        _finite_errors.extend(_examples[_column].dropna().tolist())
    _max_manual_error = max(_finite_errors) if _finite_errors else 0.0
    _grain_row = _grain.iloc[0]
    _banned_main = {"ride_ratio", "dbno_ratio", "team_kills", "team_damage", "team_assists", "team_avg_survival"}
    _registry_names = set(df_dict["feature_name"])
    _base_dictionary = df_dict[df_dict["group"] != "historical"]
    _gate_rows = [
        {"Kiểm tra": "Bảo toàn dòng", "Kỳ vọng": _n_clean, "Thực tế": base_rows, "Kết luận": "Đạt" if base_rows == _n_clean else "Không đạt"},
        {"Kiểm tra": "row_id không null", "Kỳ vọng": 0, "Thực tế": int(_grain_row["null_row_ids"]), "Kết luận": "Đạt" if int(_grain_row["null_row_ids"]) == 0 else "Không đạt"},
        {"Kiểm tra": "row_id duy nhất", "Kỳ vọng": base_rows, "Thực tế": int(_grain_row["unique_row_ids"]), "Kết luận": "Đạt" if int(_grain_row["unique_row_ids"]) == base_rows else "Không đạt"},
        {"Kiểm tra": "Đối soát numbered parts", "Kỳ vọng": base_rows, "Thực tế": int(_parts["row_count_from_parts"]), "Kết luận": "Đạt" if int(_parts["row_count_from_parts"]) == base_rows else "Không đạt"},
        {"Kiểm tra": "Feature Registry cơ sở", "Kỳ vọng": "26 feature không historical, đủ metadata", "Thực tế": f"{len(_base_dictionary)} cơ sở; {len(df_dict)-len(_base_dictionary)} khai báo historical cho NB08", "Kết luận": "Đạt" if len(_base_dictionary) == 26 and not _base_dictionary[["formula", "unit", "missing_semantics", "allowed_tasks"]].isna().any().any() else "Không đạt"},
        {"Kiểm tra": "Không có feature core bị cấm", "Kỳ vọng": "Không có", "Thực tế": ", ".join(sorted(_registry_names & _banned_main)) or "Không có", "Kết luận": "Đạt" if not (_registry_names & _banned_main) else "Không đạt"},
        {"Kiểm tra": "S1 không chứa rate theo survival", "Kỳ vọng": "False", "Thực tế": str(bool({"kills_per_minute", "damage_per_minute"} & set(registry.get_allowed_features("s1")))), "Kết luận": "Đạt" if not ({"kills_per_minute", "damage_per_minute"} & set(registry.get_allowed_features("s1"))) else "Không đạt"},
        {"Kiểm tra": "Đối chiếu công thức", "Kỳ vọng": "Sai số <= 1e-12", "Thực tế": _max_manual_error, "Kết luận": "Đạt" if _max_manual_error <= 1e-12 else "Không đạt"},
        {"Kiểm tra": "Ba hình PNG", "Kỳ vọng": "3/3", "Thực tế": sum(path.is_file() for path in (_fig_features, _fig_modes, _fig_coverage)), "Kết luận": "Đạt" if all(path.is_file() for path in (_fig_features, _fig_modes, _fig_coverage)) else "Không đạt"},
    ]
    _df_gate = pd.DataFrame(_gate_rows)
    print("--- BẢNG 03-K: KỲ VỌNG / THỰC TẾ HOÀN TẤT NB03 ---")
    print(_df_gate.to_string(index=False))
    display(_df_gate)
    if (_df_gate["Kết luận"] != "Đạt").any():
        raise RuntimeError("Kiểm tra NB03 chưa đạt; sửa các dòng Không đạt trước khi mở Notebook 04.")

    _core_stage_artifacts = {
        "player_match_base": base_pq,
        "feature_dictionary": dict_csv,
        "feature_validation": val_csv,
        "base_schema": schema_json,
        "parts_manifest": parts_manifest_json,
    }
    ckpt_mgr.commit(
        stage="player_match_base",
        signature=cfg["features"]["feature_versions"]["feature_version"],
        artifacts=_core_stage_artifacts,
    )

    _nb03_artifacts = {
        **_core_stage_artifacts,
        "feature_distributions": _fig_features,
        "mode_distribution": _fig_modes,
        "task_coverage": _fig_coverage,
    }
    _handover = []
    for _name, _path in _nb03_artifacts.items():
        _handover.append({
            "Artifact": _name,
            "Đường dẫn": str(_path.relative_to(PROJECT_ROOT)),
            "Bytes": _path.stat().st_size,
            "SHA256": hash_file(_path)[:16] + "...",
            "Consumer": "04_combat_timing.ipynb" if _name == "player_match_base" else "Kiểm toán và báo cáo",
        })
    print("\n--- BẢNG 03-L: ARTIFACT VÀ BÀN GIAO ---")
    print(pd.DataFrame(_handover).to_string(index=False))
    display(pd.DataFrame(_handover))
    print("\nKiểm tra NB03 hoàn tất, chưa chứng nhận G3. Tiếp theo: mở 04_combat_timing.ipynb; không tính lại base features.")
    _pubg_stage_artifacts = _nb03_artifacts
    from src.utils.logging import write_handover, write_figure_metadata
    _pubg_checkpoint.commit('notebook/03_build_player_match.ipynb', _pubg_signature, _pubg_stage_artifacts, writer_id=_pubg_writer)
    write_handover(paths["manifests"] / "notebook_handover.csv", stage='03_build_player_match.ipynb', artifacts=_pubg_stage_artifacts, status="completed", writer_id=_pubg_writer, version=_pubg_signature, next_step='04_combat_timing.ipynb')
    write_figure_metadata(paths["manifests"] / "figure_metadata.csv", stage='03_build_player_match.ipynb', artifacts=_pubg_stage_artifacts, details=globals().get("_pubg_figure_details", {}))

except (Exception, KeyboardInterrupt) as _nb02_error:
    _pubg_progress['03_build_player_match.ipynb'] = -1
    try:
        import duckdb
        _pubg_checkpoint.record_failure('notebook/03_build_player_match.ipynb', _pubg_signature, str(_nb02_error), resource_limited=isinstance(_nb02_error, (MemoryError, duckdb.OutOfMemoryException)), writer_id=_pubg_writer)
    except Exception as _nb02_checkpoint_error:
        print("Không ghi được trạng thái lỗi; giữ lỗi gốc và kiểm tra manifest trước khi tiếp quản:", _nb02_checkpoint_error)
    raise


_pubg_progress['03_build_player_match.ipynb'] = 20
